In [0]:
##### Imports ######
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

emp_data = [(1,'manish',50000,'IT','m'),
(2,'vikash',60000,'sales','m'),
(3,'raushan',70000,'marketing','m'),
(4,'mukesh',80000,'IT','m'),
(5,'priti',90000,'sales','f'),
(6,'nikita',45000,'marketing','f'),
(7,'ragini',55000,'marketing','f'),
(8,'rashi',100000,'IT','f'),
(9,'aditya',65000,'IT','m'),
(10,'rahul',50000,'marketing','m'),
(11,'rakhi',50000,'IT','f'),
(12,'akhilesh',90000,'sales','m')]

emp_schema = ['id','name','salary','dept','gender']
emp_df = spark.createDataFrame(data=emp_data,schema=emp_schema)
emp_df.select('id','name','salary','gender','dept')
emp_df.show()




### Group By

In [0]:
emp_df.groupBy('dept').agg(sum('salary')).show()

#### Window Functions

In [0]:
from pyspark.sql.window import Window

window =Window.partitionBy('dept').orderBy(desc('salary'))
emp_df.withColumn('total_sal',sum(col('salary')).over(window)).show(truncate=False)

In [0]:


window =Window.partitionBy('dept').orderBy('salary')
emp_df.withColumn('row_number',row_number().over(window))\
    .withColumn("rank",rank().over(window))\
    .withColumn("dense_rank",dense_rank().over(window))\
     .show(truncate=False)

In [0]:
product_data = [
(1,"iphone","01-01-2023",1500000),
(2,"samsung","01-01-2023",1100000),
(3,"oneplus","01-01-2023",1100000),
(1,"iphone","01-02-2023",1300000),
(2,"samsung","01-02-2023",1120000),
(3,"oneplus","01-02-2023",1120000),
(1,"iphone","01-03-2023",1600000),
(2,"samsung","01-03-2023",1080000),
(3,"oneplus","01-03-2023",1160000),
(1,"iphone","01-04-2023",1700000),
(2,"samsung","01-04-2023",1800000),
(3,"oneplus","01-04-2023",1170000),
(1,"iphone","01-05-2023",1200000),
(2,"samsung","01-05-2023",980000),
(3,"oneplus","01-05-2023",1175000),
(1,"iphone","01-06-2023",1100000),
(2,"samsung","01-06-2023",1100000),
(3,"oneplus","01-06-2023",1200000)
 ]

product_schema = ['product_id','product_name','sales_date','sales']
product_df = spark.createDataFrame(data=product_data,schema=product_schema)
product_df.show()

In [0]:
window = Window.partitionBy('product_id').orderBy('sales_date')
Last_month_df = product_df.withColumn('previsous_month_sales',lag(col('sales'),1).over(window))
Last_month_df.show()


In [0]:
Last_month_df.withColumn("per_loss_gain",
                         round(((col('sales')-col('previsous_month_sales'))/col('sales'))*100,2)).show()

In [0]:
#### Q1. Data:-
product_data = [
(2,"samsung","01-01-1995",11000),
(1,"iphone","01-02-2023",1300000),
(2,"samsung","01-02-2023",1120000),
(3,"oneplus","01-02-2023",1120000),
(1,"iphone","01-03-2023",1600000),
(2,"samsung","01-03-2023",1080000),
(3,"oneplus","01-03-2023",1160000),
(1,"iphone","01-01-2006",15000),
(1,"iphone","01-04-2023",1700000),
(2,"samsung","01-04-2023",1800000),
(3,"oneplus","01-04-2023",1170000),
(1,"iphone","01-05-2023",1200000),
(2,"samsung","01-05-2023",980000),
(3,"oneplus","01-05-2023",1175000),
(1,"iphone","01-06-2023",1100000),
(3,"oneplus","01-01-2010",23000),
(2,"samsung","01-06-2023",1100000),
(3,"oneplus","01-06-2023",1200000)
]

product_schema=["product_id","product_name","sales_date","sales"]

product_df = spark.createDataFrame(data=product_data,schema=product_schema)

product_df.show()


In [0]:
window = Window.partitionBy('product_id').orderBy('sales_date').rowsBetween(Window.unboundedPreceding,Window.unboundedFollowing)


In [0]:
product_df.withColumn("first_sales",first(col('sales')).over(window))\
            .withColumn("last_sale",last(col('sales')).over(window)).show()

In [0]:
emp_data = [(1,"manish","11-07-2023","10:20"),
        (1,"manish","11-07-2023","11:20"),
        (2,"rajesh","11-07-2023","11:20"),
        (1,"manish","11-07-2023","11:50"),
        (2,"rajesh","11-07-2023","13:20"),
        (1,"manish","11-07-2023","19:20"),
        (2,"rajesh","11-07-2023","17:20"),
        (1,"manish","12-07-2023","10:32"),
        (1,"manish","12-07-2023","12:20"),
        (3,"vikash","12-07-2023","09:12"),
        (1,"manish","12-07-2023","16:23"),
        (3,"vikash","12-07-2023","18:08")]

emp_schema = ["id", "name", "date", "time"]
emp_df = spark.createDataFrame(data=emp_data, schema=emp_schema)

emp_df.show()

In [0]:
emp_df.withColumn("timestamp",
                  from_unixtime(unix_timestamp(expr("CONCAT(date, ' ',time)"),"dd-MM-yyyy HH:mm"))).show()

In [0]:
window = Window.partitionBy("id","date").orderBy("date").rangeBetween(Window.unboundedPreceding,Window.unboundedFollowing)

In [0]:
emp_df1 = emp_df.withColumn("timestamp",
                  from_unixtime(unix_timestamp(expr("CONCAT(date, ' ',time)"),"dd-MM-yyyy HH:mm")))
emp_df1.show()

In [0]:
new_df = emp_df1.withColumn("login",first("timestamp").over(window))\
                .withColumn("logout",last("timestamp").over(window))\
                    .withColumn("login",to_timestamp("login", "yyyy-MM-dd HH:mm:ss"))\
                        .withColumn("logout",to_timestamp("logout", "yyyy-MM-dd HH:mm:ss"))\
                            .withColumn("total_time", col("logout")-col("login")).show()

In [0]:
product_data = [
(1,"iphone","01-01-2023",1500000),
(2,"samsung","01-01-2023",1100000),
(3,"oneplus","01-01-2023",1100000),
(1,"iphone","01-02-2023",1300000),
(2,"samsung","01-02-2023",1120000),
(3,"oneplus","01-02-2023",1120000),
(1,"iphone","01-03-2023",1600000),
(2,"samsung","01-03-2023",1080000),
(3,"oneplus","01-03-2023",1160000),
(1,"iphone","01-04-2023",1700000),
(2,"samsung","01-04-2023",1800000),
(3,"oneplus","01-04-2023",1170000),
(1,"iphone","01-05-2023",1200000),
(2,"samsung","01-05-2023",980000),
(3,"oneplus","01-05-2023",1175000),
(1,"iphone","01-06-2023",1100000),
(2,"samsung","01-06-2023",1100000),
(3,"oneplus","01-06-2023",1200000)
]

product_schema=["product_id","product_name","sales_date","sales"]

product_df = spark.createDataFrame(data=product_data,schema=product_schema)
product_df.show()

In [0]:
window = Window.partitionBy("product_id").orderBy("sales_date").rowsBetween(-2,0)

In [0]:
product_df.withColumn("running_sum",sum("sales").over(window)).show()

In [0]:
df= spark.range(10)
df.show()

In [0]:
# Define window specification - partition by product and year, order by date
window_spec = Window.partitionBy("product_id", "year").orderBy("sale_date").rowsBetween(Window.unboundedPreceding, Window.currentRow)

# Calculate running sales
result_df = df.withColumn("running_sales", spark_sum("sales").over(window_spec))

# Show results
result_df.select("product_name", "sale_date", "sales", "running_sales").show(truncate=False)

In [0]:
# ========================================
# DATabricks Notebook: SQL Window Function Demo
# ========================================

# Step 1: Create sample sales data
sales_data = [
    # Electronics category
    ("Electronics", "ELEC-101", 5000),  # Top seller
    ("Electronics", "ELEC-101", 4500),
    ("Electronics", "ELEC-101", 3000),
    ("Electronics", "ELEC-102", 4000),  # 2nd
    ("Electronics", "ELEC-102", 3500),
    ("Electronics", "ELEC-102", 2300),
    ("Electronics", "ELEC-103", 3000),  # 3rd
    ("Electronics", "ELEC-103", 2500),
    ("Electronics", "ELEC-103", 2000),
    ("Electronics", "ELEC-104", 1500),  # Lower
    ("Electronics", "ELEC-104", 1000),
    
    # Clothing category
    ("Clothing", "CLOTH-201", 3500),    # Top
    ("Clothing", "CLOTH-201", 3000),
    ("Clothing", "CLOTH-201", 2000),
    ("Clothing", "CLOTH-202", 2800),    # 2nd
    ("Clothing", "CLOTH-202", 2500),
    ("Clothing", "CLOTH-202", 1900),
    ("Clothing", "CLOTH-203", 2200),    # 3rd
    ("Clothing", "CLOTH-203", 2000),
    ("Clothing", "CLOTH-203", 1900),
    ("Clothing", "CLOTH-204", 1200),
    
    # Books category
    ("Books", "BOOK-301", 1800),        # Top
    ("Books", "BOOK-301", 1500),
    ("Books", "BOOK-301", 900),
    ("Books", "BOOK-302", 1600),        # 2nd
    ("Books", "BOOK-302", 1300),
    ("Books", "BOOK-302", 900),
    ("Books", "BOOK-303", 1200),        # 3rd
    ("Books", "BOOK-303", 1000),
    ("Books", "BOOK-303", 700),
    ("Books", "BOOK-304", 500)
]

# Step 2: Create DataFrame
df = spark.createDataFrame(
    sales_data, 
    ["category", "product_id", "sales"]
)

# Step 3: Create temporary view for SQL
df.createOrReplaceTempView("sales_table")

print("✅ DataFrame created with", df.count(), "rows")
df.show(100, truncate=False)


In [0]:
top_products_df = spark.sql("""
    SELECT category, product_id, total_sales
    FROM (
        SELECT category, product_id, 
               SUM(sales) AS total_sales,
               RANK() OVER (PARTITION BY category ORDER BY SUM(sales) DESC) AS rnk
        FROM sales_table
        GROUP BY category, product_id
    ) t
    WHERE rnk <= 3
    ORDER BY category, rnk
""")

top_products_df.show(truncate=False)

top_products_df.describe().show()
